In [28]:
import pandas as pd
import requests
import datetime
import numpy as np
import yfinance as yf
import math

pd.options.display.max_rows = 200

In [29]:
cookies = {
    '_gid': 'GA1.3.1023802383.1772486322',
    '_gat_gtag_UA_130714908_1': '1',
    '.AspNetCore.Antiforgery.g1ir8UL7PVw': 'CfDJ8FI-ShC1qCRDrecJJU90P_KTNar-OPe6u3OLYeQyZgSpVY0d5XmINtnwjUocFJwJvxUnG4O_4kuiGXFbrYAfR3xR4ZlN9KyncdQx-pUgK3Kp331-DiOrG8VKPt_Mwj5oI3r7FCkMF3ny0oxdZ2UI4BQ',
    '.AspNetCore.Identity.Application': 'CfDJ8FI-ShC1qCRDrecJJU90P_JIGYiw5801c8E6CkwoAg3iPamQu1rnzabql1-c3Zsf6YNLcvmsWsIAwRGwlLQ09Cwr69kFKSH6hNaMHlu3zDNoUV88VqV358lUetI3WXG2aWVNG-rWbMWsHsTSe2v6cZ-xX0MbP8Pf3uErdHOE_UiPsYJfOhc6bhBu5hFlgmF6QKVVmc3pgi4qipEUz7Gj-5HxTRFmcW96YaLJtgw88ZazGQnoIcA6pRjKRUPzbm4KT0LvoWV5cz_qF2fUELk2JhfiCu1oUgwkJ5BIupBOwtx4QG41mZ0zKqeazAyzFhgnnnF877Q5hAQrY5YWzNhnXclCgz6FyxJLc-mLYvMmyJidhgRX6NsTtVh4xsjCmqlwu82JLNuSPibtF1NAJfO2usHxDMoDX7OBtOp44ABDeE1HmKm8e5lPflvH09RoZOYaEymSyqGL9mm3fPg1yfWa6BWs2zxeZA3Bwv3j1xF_rHcMMTlDQJKRQF2-lLVM56WtJhlgAaDKkSvF7o4qCpq0B7vuXPAkHW9EbznV6TE_WqwFZGOWiC9qpRwrJxw2iVYWi9fSnRYiyOw7kK6tIeksao500-0vmmMpNZriKwSw3OZJX9w0HabTEzdlpBZRhlkzDwEhDcHWFvjcDlu7BGDml-FEQGJiYhkHfJeBlt111VWFBCTWsYR0a5dbUpo_edMH-cAepnNz7BWEGb9Fr53r4ac',
    '_ga_YH2ELJFQPC': 'GS2.3.s1772486321$o10$g1$t1772486341$j40$l0$h0',
    '_ga': 'GA1.3.1952201757.1770129456',
}

headers = {
    'accept': 'application/json, text/javascript, */*; q=0.01',
    'accept-language': 'pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7',
    'priority': 'u=1, i',
    'referer': 'https://opcoes.net.br/opcoes/estrategias/nova',
    'sec-ch-ua': '"Not:A-Brand";v="99", "Google Chrome";v="145", "Chromium";v="145"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
    'x-requested-with': 'XMLHttpRequest',
}

In [30]:


precos_teto_suno = {
    "WIZC3": 10.00, "BBSE3": 35.50, "BBAS3": 25.00, "UNIP6": 70.00, "SEER3": 14.00, "VALE3": 75.00,
    "PETR4": 34.00, "AXIA6": 43.80, "TUPY3": 21.00, "AGRO3": 27.50, "EGIE3": 28.60, "ITSA4": 9.50,
    "VAMO3": 10.90, "B3SA3": 17.00, "KLBN4": 5.60, "TTEN3": 14.70, "PRIO3": 62.75, "BRBI11": 18.00,
    "PNVL3": 12.00, "SIMH3": 10.00, "GMAT3": 7.12, "TIMS3": 18.60, "VIVA3": 25.00, "EZTC3": 13.29
}

carteira_PM = {
    "ABEV3": 12.31, "B3SA3": 11.08, "BBAS3": 21.96, "BBSE3": 32.00,
    "EGIE3": 28, "FLRY3": 12.47, "HYPE3": 22.24, "ITSA4": 9.41,
    "KLBN11": 25, "LEVE3": 28.10, "PETR4": 30.06, "TAEE11": 34.69,
    "UNIP6": 50.99, "VALE3": 58.13
}

precos_teto = {}

for k, v in carteira_PM.items():
    precos_teto[k] = precos_teto_suno.get(k, v)


def get_preco_atual(ticker):
    try:
        ativo = yf.Ticker(f"{ticker}.SA")
        data = ativo.history(period="1d")
        if data.empty:
            return None
        return float(data['Close'].iloc[-1])
    except:
        return None


precos_atuais = {}

for ticker in precos_teto.keys():
    preco = get_preco_atual(ticker)
    precos_atuais[ticker] = round(preco, 2)

precos_teto = precos_teto_suno

In [31]:
url = 'https://opcoes.net.br/listaopcoes/todas'
linhas = []

for ticker, preco_teto in precos_teto.items():
    try:
        params = {
            'idAcao': ticker.lower(),
            'dataReferencia': datetime.datetime.now().strftime('%Y-%m-%d')
        }

        response = requests.get(
            url,
            params=params,
            cookies=cookies,
            headers=headers,
            timeout=10
        )

        dados_json = response.json()['optionsChain']

        for callPut in dados_json:
            for vencimento in dados_json[callPut]:
                for _, strike in dados_json[callPut][vencimento]['strikes'].items():

                    codigo = strike[0]
                    preco_strike = _
                    ultimo_negocio = strike[2]
                    premio = strike[3]
                    volume = strike[4]

                    linhas.append({
                        'ticker': ticker,
                        'codigo': codigo,
                        'tipo': callPut,
                        'cotacao': precos_atuais.get(ticker),
                        'strike': preco_strike,
                        'premio': premio,
                        'volume': volume,
                        'preco_teto': preco_teto,
                        'vencimento': vencimento,
                        'ultimo_negocio': ultimo_negocio,
                    })

    except Exception as e:
        print(f'Erro no ticker {ticker}: {e}')


df = pd.DataFrame(linhas).dropna()

In [32]:
cols_float = ["cotacao", "strike", "premio", "volume", "preco_teto"]

for col in cols_float:
    df[col] = pd.to_numeric(
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )


cols_str = ["ticker", "codigo", "tipo"]

for col in cols_str:
    df[col] = df[col].astype("string")

df['vencimento'] = pd.to_datetime(df['vencimento'])

In [33]:
df['rentabilidade'] = round((df['premio'] / df['strike']) * 100, 2)
df['distancia'] = round(df['strike'] / df['cotacao'], 3)
df['strike - premio'] = df['strike'] - df['premio']

In [34]:
# Config

aporte = 5700

In [35]:
df['intrinsecoPUT'] = (df['strike'] - df['cotacao']).clip(lower=0)
df['extrinsecoPUT'] = abs(df['premio'] - df['intrinsecoPUT'])


df['cotas'] = np.floor((aporte / df['strike'])/100) * 100
df['premio X cotas'] = df['cotas'] * df['premio']



# Venda de PUT


In [47]:
df_filtrado_PUT = df[

    (df['tipo'] == 'PUT') &
    (df['vencimento'] <= (datetime.datetime.now() + datetime.timedelta(days=90))) &
    (df['vencimento'] >= (datetime.datetime.now() + datetime.timedelta(days=0))) &
    (df['volume'] > 0) &
    (df['strike - premio'] <= df['preco_teto']) &
    (df['rentabilidade']>1) &
    (df['distancia'] <= 0.96) # O mercado esta em alta logo quero por enquanto apenas premio, quando quiser ser exercido trocar para 0.9 até 1 que assim compro com grande chance de exercicio

][['ticker', 'codigo', 'tipo', 'cotacao', 'strike', 'distancia', 'premio', 'strike - premio', 'intrinsecoPUT', 'extrinsecoPUT',
   'rentabilidade', 'volume', 'preco_teto','cotas','premio X cotas', 'vencimento', 'ultimo_negocio',]].sort_values('rentabilidade', ascending=False)

df_filtrado_PUT.head(40)

,ticker,codigo,tipo,cotacao,strike,distancia,premio,strike - premio,intrinsecoPUT,extrinsecoPUT,rentabilidade,volume,preco_teto,cotas,premio X cotas,vencimento,ultimo_negocio
13481,B3SA3,Q175,PUT,18.42,17.53,0.952,1.04,16.49,0.0,1.04,5.93,36.0,17.0,300.0,312.0,2026-05-15,09/03/2026 15:38:38
13480,B3SA3,Q172,PUT,18.42,17.28,0.938,0.95,16.33,0.0,0.95,5.50,40.8,17.0,300.0,285.0,2026-05-15,09/03/2026 16:23:45
13478,B3SA3,Q167,PUT,18.42,16.78,0.911,0.87,15.91,0.0,0.87,5.18,41.6,17.0,300.0,261.0,2026-05-15,09/03/2026 13:44:18
13476,B3SA3,Q162,PUT,18.42,16.28,0.884,0.81,15.47,0.0,0.81,4.98,42.6,17.0,300.0,243.0,2026-05-15,09/03/2026 11:28:40
13357,B3SA3,P176,PUT,18.42,17.62,0.957,0.87,16.75,0.0,0.87,4.94,41.2,17.0,300.0,261.0,2026-04-17,09/03/2026 16:46:25
13479,B3SA3,Q170,PUT,18.42,17.03,0.925,0.84,16.19,0.0,0.84,4.93,41.1,17.0,300.0,252.0,2026-05-15,09/03/2026 16:22:39
13296,B3SA3,P168W2,PUT,18.42,16.87,0.916,0.82,16.05,0.0,0.82,4.86,42.8,17.0,300.0,246.0,2026-04-10,09/03/2026 11:33:27
5773,VALE3,Q774,PUT,80.92,77.40,0.957,3.50,73.90,0.0,3.50,4.52,34.9,75.0,0.0,0.0,2026-05-15,09/03/2026 12:10:32
13356,B3SA3,P173,PUT,18.42,17.37,0.943,0.76,16.61,0.0,0.76,4.38,41.2,17.0,300.0,228.0,2026-04-17,09/03/2026 16:37:55
2555,BBAS3,Q242,PUT,25.34,24.00,0.947,1.00,23.00,0.0,1.00,4.17,38.3,25.0,200.0,200.0,2026-05-15,09/03/2026 10:31:17


In [37]:
round(df_filtrado_PUT.describe(),2)

,cotacao,strike,distancia,premio,strike - premio,intrinsecoPUT,extrinsecoPUT,rentabilidade,volume,preco_teto,cotas,premio X cotas,vencimento
count,6.00,6.00,6.00,6.00,6.00,6.0,6.00,6.00,6.00,6.0,6.0,6.00,6
mean,34.82,33.18,0.95,0.48,32.70,0.0,0.48,1.44,21.75,35.5,100.0,47.67,2026-04-26 08:00:00
min,34.82,32.43,0.93,0.35,31.98,0.0,0.35,1.05,19.50,35.5,100.0,35.00,2026-03-27 00:00:00
25%,34.82,33.05,0.95,0.42,32.55,0.0,0.42,1.28,21.02,35.5,100.0,42.50,2026-04-11 18:00:00
50%,34.82,33.42,0.96,0.44,32.83,0.0,0.44,1.36,22.10,35.5,100.0,44.50,2026-05-01 00:00:00
75%,34.82,33.43,0.96,0.48,32.98,0.0,0.48,1.45,22.72,35.5,100.0,48.00,2026-05-15 00:00:00
max,34.82,33.43,0.96,0.71,33.08,0.0,0.71,2.12,23.20,35.5,100.0,71.00,2026-05-15 00:00:00
std,0.00,0.42,0.01,0.12,0.41,0.0,0.12,0.36,1.40,0.0,0.0,12.32,NaN


In [38]:
df_filtrado_PUT.to_html("vendaDePUT.html")
round(df_filtrado_PUT.describe(),2).to_html("vendaDePutDescribe.html")

# Venda de CALL

In [39]:
df['intrinsecoCALL'] = (df['cotacao'] - df['strike'] ).clip(lower=0)
df['extrinsecoCALL'] = abs(df['premio'] - df['intrinsecoCALL'])

df['strike + premio'] = df['strike'] + df['premio']

In [40]:
# A venda de call ainda nao esta claro para mim se vale apena, já que sou buy and foda-se, nao quero me desfazer de meus ativos
# Venderia call apenas pelo premio

df_filtrado_CALL = df[

    (df['tipo'] == 'CALL') &
    (df['vencimento'] <= (datetime.datetime.now() + datetime.timedelta(days=90))) &
    (df['vencimento'] >= (datetime.datetime.now() + datetime.timedelta(days=0))) &
    (df['volume'] > 0) &
    (df['strike + premio'] >= df['preco_teto']) &
    (df['distancia'] > 1.05) 


][['ticker', 'codigo', 'tipo', 'cotacao', 'strike', 'distancia', 'premio', 'strike + premio', 'intrinsecoCALL', 'extrinsecoCALL',
   'rentabilidade', 'volume', 'preco_teto','cotas','premio X cotas', 'vencimento', 'ultimo_negocio',]].sort_values('rentabilidade', ascending=False)

df_filtrado_CALL.head(30)

,ticker,codigo,tipo,cotacao,strike,distancia,premio,strike + premio,intrinsecoCALL,extrinsecoCALL,rentabilidade,volume,preco_teto,cotas,premio X cotas,vencimento,ultimo_negocio
7365,PETR4,E449W4,CALL,42.41,44.90,1.059,2.45,47.35,0.0,2.45,5.46,29.9,34.0,100.0,245.0,2026-05-22,09/03/2026 14:05:54
4159,VALE3,D890W4,CALL,80.92,89.00,1.100,4.77,93.77,0.0,4.77,5.36,27.6,75.0,0.0,0.0,2026-04-24,10/02/2026 17:06:50
3862,VALE3,D880W2,CALL,80.92,88.00,1.087,4.69,92.69,0.0,4.69,5.33,28.3,75.0,0.0,0.0,2026-04-10,30/01/2026 13:15:51
7309,PETR4,E450,CALL,42.41,45.00,1.061,2.30,47.30,0.0,2.30,5.11,35.7,34.0,100.0,230.0,2026-05-15,09/03/2026 16:36:48
1155,BBAS3,D270W2,CALL,25.34,26.76,1.056,1.33,28.09,0.0,1.33,4.97,27.4,25.0,200.0,266.0,2026-04-10,27/02/2026 12:16:43
7139,PETR4,D449W4,CALL,42.41,44.90,1.059,2.10,47.00,0.0,2.10,4.68,32.0,34.0,100.0,210.0,2026-04-24,09/03/2026 12:02:59
7366,PETR4,E454W4,CALL,42.41,45.40,1.071,1.83,47.23,0.0,1.83,4.03,24.2,34.0,100.0,183.0,2026-05-22,09/03/2026 14:12:13
1385,BBAS3,E268W2,CALL,25.34,26.76,1.056,1.05,27.81,0.0,1.05,3.92,25.9,25.0,200.0,210.0,2026-05-08,04/03/2026 15:46:14
7313,PETR4,E460,CALL,42.41,46.00,1.085,1.80,47.80,0.0,1.80,3.91,31.9,34.0,100.0,180.0,2026-05-15,09/03/2026 16:22:56
7310,PETR4,E452,CALL,42.41,45.25,1.067,1.75,47.00,0.0,1.75,3.87,30.2,34.0,100.0,175.0,2026-05-15,09/03/2026 16:40:50


In [41]:
round(df_filtrado_CALL.describe(),2)

,cotacao,strike,distancia,premio,strike + premio,intrinsecoCALL,extrinsecoCALL,rentabilidade,volume,preco_teto,cotas,premio X cotas,vencimento
count,691.00,691.00,691.00,691.00,691.00,691.0,691.00,691.00,691.00,691.00,691.0,691.00,691
mean,51.48,78.72,1.42,0.27,78.99,0.0,0.27,0.50,45.50,47.07,74.1,14.30,2026-04-08 15:08:35.774240256
min,13.68,14.48,1.05,0.01,14.50,0.0,0.01,0.00,14.20,9.50,0.0,0.00,2026-03-13 00:00:00
25%,25.34,32.50,1.11,0.02,32.52,0.0,0.02,0.03,32.45,25.00,0.0,0.00,2026-03-20 00:00:00
50%,42.41,51.65,1.20,0.06,51.94,0.0,0.06,0.12,40.10,34.00,100.0,1.00,2026-04-17 00:00:00
75%,80.92,100.29,1.43,0.25,100.68,0.0,0.25,0.55,52.95,75.00,100.0,10.00,2026-04-17 00:00:00
max,80.92,446.90,5.52,4.77,446.91,0.0,4.77,5.46,210.50,75.00,300.0,266.00,2026-05-22 00:00:00
std,26.29,67.58,0.65,0.54,67.54,0.0,0.54,0.85,22.03,24.68,84.5,33.55,NaN


In [42]:
df_filtrado_CALL.to_html("vendaDeCall.html")
round(df_filtrado_CALL.describe(),2).to_html("vendaDeCallDescribe.html")

# Geral

In [43]:
new_df = df[['ticker', 'codigo', 'tipo', 'cotacao', 'strike', 'distancia', 'premio', 'strike + premio','strike - premio', 'intrinsecoPUT', 'extrinsecoPUT', 'intrinsecoCALL', 'extrinsecoCALL',
   'rentabilidade', 'volume', 'preco_teto','cotas','premio X cotas', 'vencimento', 'ultimo_negocio',]].sort_values('rentabilidade', ascending=False)

new_df['cotas'] = np.floor((aporte / new_df['strike'])/100) * 100
new_df['premio X cotas'] = new_df['cotas'] * new_df['premio']

new_df[
    (new_df['vencimento'] <= (datetime.datetime.now() + datetime.timedelta(days=90))) &
    (new_df['vencimento'] >= (datetime.datetime.now() + datetime.timedelta(days=0))) &
    (new_df['tipo'] == 'PUT') &
    (new_df['ticker'].isin(['BBSE3'] )) &
    (new_df['distancia'] <= 1) 
    
  
].head(20)[['ticker', 'codigo', 'tipo', 'cotacao', 'strike', 'distancia', 'premio', 'strike - premio', 'intrinsecoPUT', 'extrinsecoPUT',
   'rentabilidade', 'volume', 'preco_teto','cotas','premio X cotas', 'vencimento', 'ultimo_negocio',]].sort_values('rentabilidade', ascending=False)


,ticker,codigo,tipo,cotacao,strike,distancia,premio,strike - premio,intrinsecoPUT,extrinsecoPUT,rentabilidade,volume,preco_teto,cotas,premio X cotas,vencimento,ultimo_negocio
675,BBSE3,P372,PUT,34.82,34.68,0.996,0.97,33.71,0.0,0.97,2.80,20.3,35.5,100.0,97.0,2026-04-17,09/03/2026 15:20:35
754,BBSE3,Q370,PUT,34.82,34.43,0.989,0.92,33.51,0.0,0.92,2.67,20.2,35.5,100.0,92.0,2026-05-15,05/03/2026 14:22:50
559,BBSE3,O372,PUT,34.82,34.68,0.996,0.88,33.80,0.0,0.88,2.54,22.8,35.5,100.0,88.0,2026-03-20,09/03/2026 16:08:52
593,BBSE3,O370W4,PUT,34.82,34.42,0.989,0.84,33.58,0.0,0.84,2.44,20.8,35.5,100.0,84.0,2026-03-27,05/03/2026 17:51:52
753,BBSE3,Q367,PUT,34.82,34.18,0.982,0.78,33.40,0.0,0.78,2.28,18.2,35.5,100.0,78.0,2026-05-15,09/03/2026 11:27:53
755,BBSE3,Q372,PUT,34.82,34.68,0.996,0.75,33.93,0.0,0.75,2.16,18.1,35.5,100.0,75.0,2026-05-15,04/03/2026 17:51:05
750,BBSE3,Q360,PUT,34.82,33.43,0.960,0.71,32.72,0.0,0.71,2.12,22.2,35.5,100.0,71.0,2026-05-15,09/03/2026 11:54:12
752,BBSE3,Q365,PUT,34.82,33.93,0.974,0.70,33.23,0.0,0.70,2.06,21.2,35.5,100.0,70.0,2026-05-15,09/03/2026 16:45:28
674,BBSE3,P367,PUT,34.82,34.18,0.982,0.69,33.49,0.0,0.69,2.02,20.2,35.5,100.0,69.0,2026-04-17,09/03/2026 16:25:12
592,BBSE3,O365W4,PUT,34.82,33.92,0.974,0.59,33.33,0.0,0.59,1.74,21.0,35.5,100.0,59.0,2026-03-27,05/03/2026 17:50:00
